In [1]:
from langchain_community.llms.mlx_pipeline import MLXPipeline
from langchain_core.prompts import PromptTemplate
import os
from dotenv import load_dotenv
load_dotenv()

/Users/anshumannarayan/Projects/phd/llm_testing/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [ ]:
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

In [3]:
from mlx_lm import load, generate

In [4]:
model, tokenizer = load("mlx-community/Josiefied-Qwen2.5-1.5B-Instruct-abliterated-v1-f16")

Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 134816.91it/s]


In [8]:
from mlx_lm.tokenizer_utils import TokenizerWrapper

In [10]:

template = """Question: {question}

Answer: Let's think step by step."""

prompt = PromptTemplate.from_template(template)

In [11]:
prompt

PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template="Question: {question}\n\nAnswer: Let's think step by step.")

In [15]:
llm = MLXPipeline.from_model_id(
    model_id = "mlx-community/quantized-gemma-2b-it",
    pipeline_kwargs = {"max_tokens": 10, "temp": 0.2}
)

chain = prompt | llm

question = "What are the key benefits of using MLX with LangChain?"

#response = generate(model, tokenizer, prompt=question, verbose=True)

Fetching 7 files: 100%|██████████| 7/7 [00:49<00:00,  7.03s/it]


In [13]:
print(chain.invoke({"question": question}))

TypeError: generate_step() got an unexpected keyword argument 'formatter'

In [17]:
help(generate)

Help on function generate in module mlx_lm.generate:

generate(model: mlx.nn.layers.base.Module, tokenizer: Union[transformers.tokenization_python.PythonBackend, mlx_lm.tokenizer_utils.TokenizerWrapper], prompt: Union[str, List[int]], verbose: bool = False, **kwargs) -> str
    Generate a complete response from the model.
    
    Args:
       model (nn.Module): The language model.
       tokenizer (PreTrainedTokenizer): The tokenizer.
       prompt (Union[str, List[int]]): The input prompt string or integer tokens.
       verbose (bool): If ``True``, print tokens and timing information.
           Default: ``False``.
       kwargs: The remaining options get passed to :func:`stream_generate`.
          See :func:`stream_generate` for more details.



In [16]:
prompt = "Write a story about Einstein"

messages = [{"role": "user", "content": prompt}]
prompt = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True,
)
text = generate(model, tokenizer, prompt=prompt, verbose=True)

Einstein was a man of many talents, but perhaps his most famous achievement was his theory of relativity. This theory, which he developed in the early 20th century, revolutionized our understanding of space, time, and gravity. It was based on the idea that space and time are not absolute, but rather are relative to the motion of an observer. This theory has been confirmed by numerous experiments and observations, and it has had a profound impact on our understanding of the universe. Einstein was also a brilliant physicist, and he made many other important contributions to the field of physics. He was awarded the Nobel Prize in Physics in 1921 for his work on the photoelectric effect. He was a man of great intellect and curiosity, and he was always eager to learn and explore new ideas. He was also a deeply spiritual man, and he believed that the universe was full of wonder and beauty. He was a man of many talents, but perhaps his most famous achievement was his theory of relativity.
Pro

In [13]:
response = generate(model, tokenizer, prompt="What is the best way to better pass prompts into a lanuage model using the mlx-lm python package?", verbose=True)

I have a prompt that I want to pass into the model, but I'm not sure how to do it. I have tried using the `prompt` parameter in the `mlx_linguistic_model.LinguisticModel` class, but I'm not sure if I'm using it correctly. I also have a prompt that I want to pass into the model, but I'm not sure how to do it. I have tried using the `prompt` parameter in the `mlx_linguistic_model.LinguisticModel` class, but I'm not sure if I'm using it correctly. I also have a prompt that I want to pass into the model, but I'm not sure how to do it. I have tried using the `prompt` parameter in the `mlx_linguistic_model.LinguisticModel` class, but I'm not sure if I'm using it correctly. I also have a prompt that I want to pass into the model, but I'm not sure how to do it. I have tried using the `prompt` parameter in the `mlx_linguistic_model.LinguisticModel` class, but I'm not sure if I'm using it correctly. I also have a prompt that I want to pass into the model
Prompt: 23 tokens, 92.364 tokens-per-sec


In [14]:
from mlx_lm import load, stream_generate
from mlx_lm.sample_utils import make_sampler

MODEL_ID = "mlx-community/Josiefied-Qwen2.5-1.5B-Instruct-abliterated-v1-f16"
model, tokenizer = load(MODEL_ID)  # load once

sampler = make_sampler(temp=0.6, top_p=0.9)
MAX_NEW_TOKENS = 256
MAX_CTX_TOKENS = 6000

messages = [{"role": "system", "content": "You are a helpful assistant."}]

def build_prompt(msgs):
    # Use model-native chat template when available
    if getattr(tokenizer, "has_chat_template", False) or getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(
            msgs,
            add_generation_prompt=True,
            return_dict=False,
        )
    # Fallback plain prompt format
    text = ""
    for m in msgs:
        text += f"{m['role'].upper()}: {m['content']}\n"
    return text + "ASSISTANT: "

def token_count(prompt):
    if isinstance(prompt, list):
        return len(prompt)
    return len(tokenizer.encode(prompt))

def trim_history(msgs, max_ctx_tokens=MAX_CTX_TOKENS):
    if len(msgs) <= 2:
        return msgs
    system = [msgs[0]] if msgs[0]["role"] == "system" else []
    tail = msgs[1:] if system else msgs[:]
    for i in range(len(tail)):
        candidate = system + tail[i:]
        if token_count(build_prompt(candidate)) <= max_ctx_tokens:
            return candidate
    return system + tail[-2:]  # hard fallback

def chat_turn(user_text, max_tokens=MAX_NEW_TOKENS):
    global messages
    messages.append({"role": "user", "content": user_text})
    messages = trim_history(messages)

    prompt = build_prompt(messages)
    answer = ""

    print("assistant> ", end="", flush=True)
    for chunk in stream_generate(
        model,
        tokenizer,
        prompt=prompt,
        max_tokens=max_tokens,
        sampler=sampler,
    ):
        print(chunk.text, end="", flush=True)
        answer += chunk.text
    print()

    messages.append({"role": "assistant", "content": answer})
    return answer

Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 170039.35it/s]


In [15]:
while True:
    q = input("you> ").strip()
    if q.lower() in {"quit", "exit"}:
        break
    if q:
        chat_turn(q)

assistant> I don't have a specific name, but you can call me Qwen. I'm an AI language model created by Alibaba Cloud. Do you have any questions or topics you'd like to discuss?
assistant> As an AI language model, I can be described as a model in the sense that I'm designed to simulate human-like conversation and text generation. I have a set of rules and algorithms that allow me to process and understand natural language, and I'm able to generate responses that are similar to those of a human. I'm also able to learn from data and improve over time, just like a model in a machine learning system.
assistant> As an AI language model, I have a large number of parameters, which are essentially the building blocks of my model. The exact number of parameters can vary depending on the specific model and its architecture, but I'm typically in the range of billions to trillions. This allows me to have a wide range of vocabulary and sentence structures in my responses.

In terms of architecture, 

KeyboardInterrupt: Interrupted by user